# TRACE: End-to-End Inference, Vector Snapping & APLS Routing

This notebook demonstrates:
1. Running inference with the trained model checkpoint.
2. Converting probability heatmaps into vector centerlines.
3. Snapping newly discovered roads into the master OpenStreetMap graph.
4. Running Dijkstra vehicle shortest-path routing over the newly completed road network!



In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import geopandas as gpd

sys.path.append("..")
from src.pipeline.inference import TRACEInferencePipeline
from src.evaluation.apls import compute_apls


## 1. Run TRACE Inference Pipeline


In [ ]:
pipeline = TRACEInferencePipeline(
    checkpoint_path="../checkpoints/best_model.pth",
    prob_threshold=0.50,
    snap_search_radius=30.0,
    spur_length_meters=15.0
)

image_path = "../data/raw/synthetic_tile_001.tif"
prior_path = "../data/raw/synthetic_tile_001.geojson"
output_geojson_path = "../data/outputs/updated_network.geojson"

unified_graph, out_gdf = pipeline.run(
    image_path=image_path,
    prior_geojson_path=prior_path,
    output_geojson_path=output_geojson_path
)

print(f"Successfully generated unified road network!")
print(f"Total Nodes: {len(unified_graph.nodes)}")
print(f"Total Edges: {len(unified_graph.edges)}")


## 2. Visualize Graph Overlay on Satellite Image


In [ ]:
import rasterio

with rasterio.open(image_path) as src:
    img = src.read([1, 2, 3]).transpose(1, 2, 0)
    transform = src.transform

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(img)

for u, v in unified_graph.edges():
    p1 = unified_graph.nodes[u]["pos_geo"]
    p2 = unified_graph.nodes[v]["pos_geo"]
    # Convert geo coordinates to pixel space for plotting
    inv_t = ~transform
    c1, r1 = inv_t * p1
    c2, r2 = inv_t * p2
    ax.plot([c1, c2], [r1, r2], color="yellow", linewidth=2.5)

ax.set_title("Unified Traversable Road Graph Overlaid on Satellite Imagery", fontsize=14)
ax.axis("off")
plt.show()


## 3. Dijkstra Shortest-Path Vehicle Routing Test


In [ ]:
nodes = list(unified_graph.nodes())
start_node = nodes[0]
end_node = nodes[-1]

if nx.has_path(unified_graph, start_node, end_node):
    path = nx.shortest_path(unified_graph, start_node, end_node, weight="weight")
    path_len = nx.shortest_path_length(unified_graph, start_node, end_node, weight="weight")
    print(f"Routing Successful!")
    print(f"Traversed {len(path)} junction nodes across {path_len:.2f} meters.")
else:
    print("Path not found - checking graph connectivity.")
